In [0]:
"""
Weekly Model Retraining Job
============================

Runs every Monday to retrain fraud detection model
with latest transaction data
"""

"""
Weekly Model Retraining Job
============================

Runs every Monday to retrain fraud detection model
"""

"""
Weekly Model Retraining Job
============================

Runs every Monday to retrain fraud detection model
"""

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import mlflow

print("="*60)
print("WEEKLY MODEL RETRAINING JOB")
print("="*60)

# ============================================
# 1. LOAD SAMPLE TRANSACTIONS FROM CSV
# ============================================

print("\nStep 1: Loading sample transactions...")

try:
    csv_path = "/Workspace/Users/molugurikoushik68@gmail.com/banking-agent/data/sample_transactions.csv"
    df = pd.read_csv(csv_path)
    print(f"✓ Loaded {len(df)} transactions from CSV")
    print(f"  Raw columns: {list(df.columns)}")
    
except Exception as e:
    print(f"✗ Error: {e}")
    df = pd.DataFrame()

# ============================================
# 2. ENGINEER FEATURES (CREATE NEW COLUMNS)
# ============================================

if len(df) > 10:
    print("\nStep 2: Engineering features from raw data...")
    
    try:
        # THIS IS THE PART I FORGOT! Create engineered features
        
        engineered = df.copy()
        
        # Feature 1: Amount (already exists, just ensure it's float)
        engineered['amount'] = df['amount'].astype(float)
        
        # Feature 2: Is new merchant?
        engineered['is_new_merchant'] = (df['merchant'] == 'Unknown').astype(int)
        
        # Feature 3: Unusual time? (10 PM - 6 AM)
        engineered['timestamp'] = pd.to_datetime(df['timestamp'])
        engineered['hour'] = engineered['timestamp'].dt.hour
        engineered['is_unusual_time'] = ((engineered['hour'] >= 22) | (engineered['hour'] < 6)).astype(int)
        
        # Feature 4: Frequency (how many times does this merchant appear?)
        merchant_counts = df['merchant'].value_counts().to_dict()
        engineered['frequency'] = df['merchant'].map(merchant_counts).fillna(1)
        
        # Feature 5: Days since last transaction
        engineered['days_since_last_transaction'] = engineered['timestamp'].diff().dt.days.fillna(0)
        
        # Feature 6: International (assume Unknown = international)
        engineered['is_international'] = (df['merchant'] == 'Unknown').astype(int)
        
        print(f"✓ Features engineered!")
        print(f"  New columns created: is_new_merchant, is_unusual_time, frequency, days_since_last_transaction, is_international")
        
        # ============================================
        # 3. TRAIN MODEL
        # ============================================
        
        print("\nStep 3: Training model...")
        
        feature_columns = ['amount', 'is_new_merchant', 'is_unusual_time', 'frequency', 'days_since_last_transaction', 'is_international']
        
        X = engineered[feature_columns].fillna(0)
        y = engineered['is_fraudulent']
        
        print(f"✓ Features ready: {len(X)} samples, {len(feature_columns)} features")
        
        # Train/test split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        # Scale
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Train model
        model = RandomForestClassifier(n_estimators=50, max_depth=8, random_state=42)
        model.fit(X_train_scaled, y_train)
        
        # Evaluate
        train_acc = model.score(X_train_scaled, y_train)
        test_acc = model.score(X_test_scaled, y_test)
        
        print(f"✓ Model trained")
        print(f"  Training accuracy: {train_acc:.1%}")
        print(f"  Testing accuracy: {test_acc:.1%}")
        
        # ============================================
        # 4. LOG TO MLFLOW WITH SIGNATURE
        # ============================================
        
        print("\nStep 4: Logging to MLflow...")
        
        mlflow.set_experiment("/Users/molugurikoushik68@gmail.com/banking-agent/fraud-detection")
        
        with mlflow.start_run(run_name="weekly_retrain"):
            # Log parameters
            mlflow.log_param("n_estimators", 50)
            mlflow.log_param("max_depth", 8)
            mlflow.log_param("train_samples", len(X_train))
            mlflow.log_param("test_samples", len(X_test))
            mlflow.log_param("scaler", "StandardScaler")
            
            # Log metrics
            mlflow.log_metric("train_accuracy", train_acc)
            mlflow.log_metric("test_accuracy", test_acc)
            mlflow.log_metric("generalization_gap", train_acc - test_acc)
            
            # FIX: Create input example for signature
            input_example = X_test.head(1)
            
            # Log model WITH signature
            mlflow.sklearn.log_model(
                model, 
                artifact_path="fraud_detector_model",
                input_example=input_example,  # ← THIS IS THE FIX!
                registered_model_name="fraud_detector"
            )
        
        print("✓ Model logged to MLflow")
        
    except Exception as e:
        print(f"✗ Failed: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"⚠️ Not enough data ({len(df)} transactions)")

print("\n" + "="*60)
print("✓ WEEKLY MODEL RETRAINING COMPLETE")
print("="*60)